# KrizKalkan AI · M5 — Türkçe Çıkarım (NLI) Modeli**M3 defteri bittikten SONRA çalıştırın.** Settings → Accelerator: **GPU T4 x2**.## Neden bu model gerekli`docs/metrikler/m5.md` ölçümü şunu gösterdi: gömme tabanlı geri getirmesıralamada çok iyi (**Recall@5 = 0,964**) ama **karar vermede kullanılamaz.**Üç istatistik denendi, hiçbirinde pozitifler negatiflerden ayrışmadı. En yüksekskorlu alakasız sorgu resmî bir AFAD duyurusuydu (0,890): havuza konu olarakgerçekten benzer, ama aynı iddia değil.Bu model o ayrımı yapar.| | ||---|---|| Veri | SNLI-TR · 550.152 satır (150.000'e dengeli altörnekleme) || Omurga | XLM-RoBERTa base || Süre | ~3–4 saat / 2 epok (T4) || Rapor hedefi | 3 sınıf doğruluk ≥ 0,80 || Kabul kapısı | doğruluk ≥ 0,70 — altındaysa ağırlık yüklenmez |

In [ ]:
# 1 · Depo ve bağımlılıklar!git clone -q https://github.com/devfurkank/KrizKalkanAI.git /kaggle/working/repo || echo "depo zaten var"%cd /kaggle/working/repo!git checkout -q feat/modeller && git pull -q!pip install -q "transformers>=4.46" "datasets>=3.1" "huggingface-hub>=0.26" onnx onnxruntime 2>&1 | tail -2import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK ⚠️")

In [ ]:
# 2 · Eğitim — 2 epok, 150k dengeli örnek!python scripts/train/m5_nli.py --epok 2 --yigin 32 --lr 2e-5 --ornek 150000 \    --cikti /kaggle/working/m5_nli

## Dışa aktarımModel int8 ONNX'e çevrilir (1112 MB → ~279 MB). `dynamo=False` bilinçli:yeni torch.export tabanlı aktarıcının grafı, onnxruntime'ın niceleme öncesişekil çıkarımını düşürüyor.

In [ ]:
# 3 · int8 ONNX dışa aktarımı!python scripts/train/m5_nli.py --disa-aktar /kaggle/working/m5_nli \    --onnx-cikti /kaggle/working/m5_nli_onnx!ls -lh /kaggle/working/m5_nli_onnx

In [ ]:
# 4 · Model kartını yaz — kabul kapısı bunu okur# Kart olmadan ağırlık YÜKLENMEZ; ölçülmemiş model üretime giremez.import json, syssys.path.insert(0, "libs/krizkalkan-core/src")from pathlib import Pathfrom krizkalkan_core.models.cards import Measurement, ModelCardrapor = json.loads(Path("/kaggle/working/m5_nli/egitim_raporu.json").read_text())son = rapor["gecmis"][-1]kart = ModelCard(    name="m5_nli",    module="M5",    title="Türkçe Çıkarım Modeli (NLI-TR)",    version="0.1.0",    base_model=rapor["omurga"],    purpose=(        "Kullanıcının iddiası ile havuzdaki kaydın AYNI iddia olup olmadığına karar verir. "        "Geri getirme aday üretir, karar bu modelindir."    ),    training_data=[f"SNLI-TR (NLI-TR) — {rapor['egitim_satir']:,} satır, sınıf dengeli altörnekleme"],    training_procedure="XLM-R base üzerine 3 sınıflı dizi-çifti sınıflandırma başlığı; erken durdurma.",    hyperparameters=rapor["hiperparametreler"],    split_strategy="SNLI-TR'nin kendi eğitim/doğrulama bölünmesi",    measurements=[        Measurement("dogruluk", son["dogruluk"], "SNLI-TR doğrulama", rapor["dogrulama_satir"]),        Measurement("makro_f1", son["makro_f1"], "SNLI-TR doğrulama", rapor["dogrulama_satir"]),    ],    known_limits=[        "SNLI-TR makine çevirisiyle üretilmiştir ve çeviri gürültüsü taşır; "        "kısa, genel cümlelerden oluşur.",        "Alan farkı: kriz iddiaları ve DMM kayıtları daha uzun ve kurumsal dildedir. "        "Alan içi başarım scripts/eval/m5_knowledge.py ile AYRICA ölçülmelidir — "        "SNLI doğruluğu tek başına alan başarımını temsil etmez.",        "Bölgesel ağız ve Türkçe dışı diller ölçülmedi.",    ],    ethical_notes=[        "Yanlış 'aynı iddia' kararı, kullanıcıya resmî kaynağın onu yalanladığını "        "söylemek demektir; eşik (0,75) bu yüzden kesinlik lehine ayarlıdır.",    ],    out_of_scope=["Doğruluk hükmü vermek", "Havuzda karşılığı olmayan iddiaları yanlış saymak"],    license="Model: MIT · Veri: SNLI türevi (araştırma)",)kart.save(Path("/kaggle/working/m5_nli_onnx"))print(kart.to_markdown()[:900])print("\nKABUL KAPISI:", "✓ GEÇER" if son["dogruluk"] >= 0.70 else "🔴 GEÇMEZ — ağırlık yüklenmeyecek")

## Çıktıyı indirme`/kaggle/working/m5_nli_onnx/` içeriğini (`model.onnx`, `tokenizer.json`,`kart.json`) Kaggle Dataset olarak yayımlayın, yerelde `models/m5_nli/`altına açın.Doğrulama:```bashKK_MODELS=on make model-durum````m5_nli` satırı **hazır** demiyorsa sebep yazılıdır (kart yok, eşik altı,dosya eksik). Sistem her durumda kural tabanlı yolla çalışmaya devam eder.Sonra alan içi ölçüm:```bashKK_MODELS=on python scripts/eval/m5_knowledge.py```

---## M3 ağırlığı bu oturumda da değerlendirilebilirM3 oturumu kapandıysa ve ağırlık Kaggle Dataset olarak kaydedildiyse, girdiolarak ekleyip Kural 0 eğrisini burada çıkarabilirsiniz — eğitim gerekmez:```bashpython scripts/eval/m3_text.py --kontrol-noktasi /kaggle/input/<dataset-adı>/m3_asama2```